In [0]:

df = spark.table("workspace.default.movies")

In [0]:
df.show()

####INTERVIEW ANSWER :
explain() is used to view the logical and physical execution plans of a Spark DataFrame query. It helps understand how Spark executes transformations, identify bottlenecks such as shuffles and full scans, verify optimizations like predicate pushdown and partition pruning, and tune query performance without actually executing the job.

#####When Should You Use explain()?
Use it when:
✅ A Spark job is running slowly <br>
✅ You want to understand the execution plan <br>
✅ You are optimizing joins <br>
✅ You need to verify predicate pushdown <br>
✅ You want to check partition pruning <br>
✅ You suspect unnecessary shuffles <br>
✅ You are learning how Spark executes transformations <br>


##### Important Note
`explain()` 
does not execute the query.
Only shows the plan.


In [0]:
from pyspark.sql import functions as F
df_narrow = df.select("title","studio", "imdb_rating").filter(F.col("release_year") > 2010)
# Till the above line it will just build physical plan it will start execution

df_narrow.explain()


####Why use explain()?

#####1. Understand Query Execution
You can see:

- Filters (Filter) <br>
- Column selections (Project) <br>
- Joins (BroadcastHashJoin, SortMergeJoin) <br>
- Aggregations (HashAggregate) <br>
- Scans (FileScan)​‌ <br>
- Example: `df.join(df2, "id").explain() `

##### You can verify whether Spark is using:

- Broadcast Join
- Sort Merge Join
- Shuffle operations

#####2. Performance Tuning
explain() helps identify expensive operations like:
- Unnecessary Shuffles <br>
Exchange hashpartitioning(...) <br>
Exchange indicates data movement across the cluster, which is costly. <br>



- Full Table Scans <br>
FileScan parquet <br>
You can check whether partition pruning is happening.<br>
For example:<br>
`df.filter(F.col("year") == 2023).explain()` <br>
If partitions are being pruned, Spark reads fewer files. <br>


#####3. Verify Optimizations
Spark's Catalyst Optimizer rewrites your query. <br>
Example:` df.filter(F.col("salary") > 1000).select("name")` <br>
Spark may push the filter closer to the data source: <br>
PushedFilters: [GreaterThan(salary,1000)] <br>

This is called Predicate Pushdown, which improves performance.





####Different Types of Explain Plans


#####Simple Plan (default)
`df.explain()`
Displays the physical execution plan.

#####Extended Plan
`df.explain(True)`
or
`df.explain(mode="extended")`

Shows:
Parsed Logical Plan
Analyzed Logical Plan
Optimized Logical Plan
Physical Plan

Useful for advanced debugging.

#####Formatted Plan
`df.explain(mode="formatted")`
Produces a more readable output:
Good for understanding complex queries.

#####Cost-Based Plan
`df.explain(mode="cost")`
Shows optimizer statistics when available.


When Should You Use explain()?
Use it when:
- ✅ A Spark job is running slowly
- ✅ You want to understand the execution plan
- ✅ You are optimizing joins
- ✅ You need to verify predicate pushdown
- ✅ You want to check partition pruning
- ✅ You suspect unnecessary shuffles
- ✅ You are learning how Spark executes transformations

Important Note
explain() does not execute the query.
No execution happens here.
Only shows the plan.





In [0]:
df_narrow.explain("extended")

# Parsed logical plan is Unresolved Logical plan
# Project means select

# Analyzed Logical Plan is  Resolved  Logical plan
# Project means select 


# == Physical Plan ==    this physical plan is "BEST PHYSICAL PLAN"
# PhotonResultStage
# photon is actual executor(Vectorized Query Executor)




In [0]:
from pyspark.sql import functions as F

df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") > 2010)

df_narrow.explain("formatted") 
# Readable format; shows the physical execution plan in a structured layout

In [0]:
df_narrow.explain()

In [0]:
df_narrow.explain("extended")

#### Spark only executes the code when the action is triggered, if the action is not hit then it only creates Physical Plan. This Physical Plan gets executed when the action is triggered.

In [0]:
revenue_df = df.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))
revenue_df.show(3)

- A transformation is an operation that defines a new dataset from an existing one but does not execute immediately.

- An action is an operation that triggers the actual execution of all pending transformations and return a result to the driver program or writes data to storage.




#####Why Spark Uses Lazy Evaluation
######1. Query Optimization​‌ <br>
Suppose you write:​‌ <br>
df.filter(F.col("year") > 2023) \ <br>
.filter(F.col("rating") > 8)​‌ <br>

Spark may optimize it into:​‌ <br>
(year > 2023) AND (rating > 8)​‌ <br>
instead of performing two separate filters.​‌ <br>

######2. Predicate Pushdown​‌ <br>
df.filter(F.col("year") == 2023)​‌ <br>
Spark can push the filter down to the data source:​‌ <br>
Read only records where year = 2023​‌ <br>
instead of reading the entire dataset.​‌ <br>

######3. Reduced Data Movement <br>
- Spark can rearrange operations to minimize shuffles and network traffic. <br>
- Since it knows the complete workflow before execution, it can choose a more efficient strategy. <br>

######Interview Answer
Spark uses lazy evaluation by delaying the execution of transformations such as filter(), select(), and join(). Instead of executing them immediately, Spark builds a logical execution plan (DAG). When an action like show(), count(), or collect() is called, Spark optimizes the plan using the Catalyst Optimizer, generates a physical execution plan, and then executes it. This improves performance by enabling optimizations such as predicate pushdown, filter merging, and reduced shuffling.

## Benefits of Lazy Evaluation in Spark

Lazy evaluation is one of the main reasons Spark can process large datasets efficiently.

### 1. Query Optimization

Since Spark sees the entire chain of transformations before executing anything, it can optimize the query using the **Catalyst Optimizer**.

```python
df.filter(F.col("age") > 18) \
  .filter(F.col("salary") > 50000)
```

Spark may combine these into:

```text
(age > 18) AND (salary > 50000)
```

✅ Fewer operations  
✅ Faster execution

---

### 2. Predicate Pushdown

Spark can push filters down to the data source.

```python
df.filter(F.col("year") == 2025)
```

Instead of:

```text
Read Entire Dataset
↓
Apply Filter
```

Spark may do:

```text
Read Only Rows Where year = 2025
```

✅ Less I/O  
✅ Less memory usage

---

### 3. Reduced Data Movement (Shuffle)

Shuffling data across executors is expensive.

Since Spark knows the complete execution plan beforehand, it can choose an efficient strategy and minimize unnecessary shuffles.

✅ Less network traffic  
✅ Better cluster performance

---

### 4. Eliminates Unnecessary Computations

Example:

```python
df.select("id", "name") \
  .filter(F.col("id") > 100)
```

Spark knows that only the required columns are needed and avoids reading unnecessary data.

✅ Reads less data  
✅ Saves CPU and memory

---

### 5. Builds an Efficient DAG

Instead of executing every transformation immediately, Spark creates a **DAG (Directed Acyclic Graph)**.

```text
Read Data
   ↓
Filter
   ↓
Select
   ↓
Group By
```

When an action is called, Spark executes the optimized DAG.

✅ Better execution planning  
✅ Better parallelization

---

### 6. Faster Overall Job Execution

Without lazy evaluation:

```text
filter()
execute

select()
execute

groupBy()
execute
```

With lazy evaluation:

```text
filter()
select()
groupBy()

show()  ← Execute once after optimization
```

✅ Fewer passes over data  
✅ Better performance

---

### 7. Fault Tolerance Through Lineage

Spark records all transformations in the DAG (lineage).

```text
Raw Data
   ↓
Filter
   ↓
Aggregate
```

If a partition is lost, Spark can recompute it from the lineage instead of rerunning the entire job.

✅ Easy recovery from failures  
✅ Reliable distributed processing

---

### Interview Answer

> Spark uses lazy evaluation to delay the execution of transformations until an action is called. This allows Spark to optimize the complete execution plan, reduce shuffling, apply predicate pushdown, eliminate unnecessary computations, improve performance, and provide fault tolerance through lineage.


## Interview Answer: What is a Narrow Transformation?

A **Narrow Transformation** is a transformation where **each output partition depends on only one input partition**. Since data does not need to move across partitions, Spark can execute the transformation within the same partition.

### Key Characteristics

- No data shuffle required
- Faster execution
- Less network I/O
- Typically executed within the same stage

### Examples

```python
select()
filter()
withColumn()
drop()
map()
flatMap()
```

### Example

```python
df.filter(F.col("age") > 18)
```

Each partition is filtered independently without exchanging data with other partitions.

### Interview Answer

> A narrow transformation is a Spark transformation in which each output partition depends on a single input partition. Since data remains within the same partition, no shuffle occurs, making narrow transformations faster and more efficient. Common examples include `filter()`, `select()`, and `withColumn()`.

---

## Interview Answer: What is a Wide Transformation?

A **Wide Transformation** is a transformation where **one output partition depends on data from multiple input partitions**. To achieve this, Spark must redistribute data across the cluster, which results in a **shuffle**.

### Key Characteristics

- Requires data shuffle
- Involves network communication between executors
- More expensive than narrow transformations
- Creates stage boundaries in Spark jobs

### Examples

```python
groupBy()
join()            # Non-broadcast join
distinct()
orderBy()
repartition()
dropDuplicates()
```

### Example

```python
df.groupBy("department").count()
```

Spark must bring all records belonging to the same department together, causing a shuffle.

### Interview Answer

> A wide transformation is a Spark transformation in which an output partition depends on multiple input partitions. Since Spark needs to redistribute data across the cluster, a shuffle occurs. Wide transformations are more expensive than narrow transformations and often create new stages. Common examples include `groupBy()`, `orderBy()`, `distinct()`, and non-broadcast `join()` operations.

---

## Easy Way to Remember

```text
Narrow Transformation
Input Partition 1 --> Output Partition 1
Input Partition 2 --> Output Partition 2
(No Shuffle)

Wide Transformation
Input Partition 1 --\
Input Partition 2 ----> Output Partition
Input Partition 3 --/
(Shuffle Required)
```

### One-Line Interview Difference

> In a narrow transformation, each output partition depends on a single input partition and no shuffle occurs. In a wide transformation, output partitions depend on multiple input partitions, requiring a shuffle and data movement across the cluster.

In [0]:
from pyspark.sql import Window

# One-time shuffle:
base = df.repartition(6, "studio")

# Reuse partitioning on the smae detailed rows:
agg = base.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))

ranked = base.withColumn("rnk", F.row_number().over(Window.partitionBy("studio").orderBy(F.col("revenue"))))




In [0]:
df = spark.table("workspace.default.movies")
display(df.limit(5))

In [0]:
%sql
select distinct studio from workspace.default.movies;

## Repartition in Spark

`repartition()` is used to **increase or decrease the number of partitions** in a DataFrame or RDD. It redistributes data across the cluster and **always performs a shuffle**, making it a **wide transformation**.

### Syntax

```python
df.repartition(num_partitions)
```

```python
df.repartition(num_partitions, "column_name")
```

---

## How Repartition Works Internally

The behavior of `repartition()` depends on whether you specify a partitioning column (key) or not.

### 1. Repartition Without a Key

```python
df.repartition(10)
```

Spark uses **Round-Robin Partitioning**.

```text
Input Data

Row1 → Partition 1
Row2 → Partition 2
Row3 → Partition 3
Row4 → Partition 4
...
```

Rows are distributed as evenly as possible across the target partitions.

### Characteristics

- Uses Round-Robin partitioning
- Data is evenly distributed
- Useful for increasing parallelism
- Helps reduce skew in many scenarios
- Requires a full shuffle

### Example

```python
movies_df.repartition(10)
```

Creates 10 evenly distributed partitions across the cluster.

---

### 2. Repartition With a Key

```python
df.repartition(10, "department")
```

Spark uses **Hash Partitioning**.

```text
hash(department) % 10
```

Rows having the same key value are sent to the same partition.

Example:

```text
department = HR
hash(HR) % 10 = 3

department = IT
hash(IT) % 10 = 7
```

All HR rows go to Partition 3 and all IT rows go to Partition 7.

### Characteristics

- Uses Hash Partitioning
- Same key values end up in the same partition
- Useful before joins and aggregations
- Requires a full shuffle

### Example

```python
employees_df.repartition(10, "department")
```

This makes subsequent operations such as:

```python
groupBy("department")
join(...)
```

more efficient because related records are already colocated.

---

## Why Does Repartition Cause a Shuffle?

When repartitioning, Spark needs to move records across executors to create the new partition layout.

```text
Before

Partition 1 : A, B
Partition 2 : C, D
Partition 3 : E, F

       Shuffle

After repartition(2)

Partition 1 : A, C, E
Partition 2 : B, D, F
```

Since output partitions depend on data from multiple input partitions, Spark inserts an **Exchange** operation.

You can see this in the execution plan:

```python
df.repartition(10).explain()
```

Output:

```text
Exchange RoundRobinPartitioning(10)
```

and

```python
df.repartition(10, "department").explain()
```

Output:

```text
Exchange hashpartitioning(department, 10)
```

---

## Repartition vs Coalesce

### Repartition

```python
df.repartition(10)
```

- Increase or decrease partitions
- Always shuffles data
- Better load balancing
- Wide transformation

### Coalesce

```python
df.coalesce(2)
```

- Mainly decreases partitions
- Tries to avoid shuffle
- Faster than repartition
- Usually a narrow transformation

---

## Interview Answer

> Repartition is a Spark transformation used to change the number of partitions in a DataFrame or RDD. It is a wide transformation because it always performs a shuffle. If no partitioning key is provided, Spark uses **Round-Robin Partitioning** to distribute data evenly across partitions. If a column is specified, Spark uses **Hash Partitioning**, where rows with the same key value are placed in the same partition. Repartition is commonly used to improve parallelism, balance data distribution, and optimize joins and aggregations.

In [0]:
# rep by key
rep_by_key = df.repartition(6, "studio")
rep_by_key.explain("formatted")



In [0]:
# if we dont specify key(COLUMN NAME) then it will do in round robin fashion
rep_rr = df.repartition(6, "studio")
rep_rr.explain("formatted")


## Flow of Execution Plan (Hinglish)

### (1) PhotonScan parquet workspace.default.movies

```text
PhotonScan parquet workspace.default.movies
```

👉 Spark movies table ko Parquet/Delta files se read kar raha hai.

Simple words mein:

```text
Storage (S3/Delta)
        ↓
Read Data
```

---

### (2) PhotonShuffleExchangeSink

```text
Arguments: hashpartitioning(studio, 6)
```

👉 Sabse important step.

Tumne likha:

```python
df.repartition(6, "studio")
```

Isliye Spark:

```text
hash(studio) % 6
```

calculate karke records ko 6 partitions mein distribute karega.

Example:

```text
Disney → Partition 2
Marvel → Partition 4
Sony   → Partition 1
```

Same studio wale records same partition mein jayenge.

✅ Hash Partitioning

✅ Shuffle Start

✅ Wide Transformation

---

### (3) PhotonShuffleMapStage

```text
Arguments: REPARTITION_BY_NUM
```

👉 Yahan actual shuffle perform ho raha hai.

Spark records ko old partitions se uthakar naye partitions mein bhej raha hai.

```text
Old Partitions
      ↓
Shuffle
      ↓
6 New Partitions
```

---

### (4) PhotonShuffleExchangeSource

```text
PhotonShuffleExchangeSource
```

👉 Shuffle complete ho gaya.

Ab Spark newly created partitions ko read kar raha hai.

```text
Shuffle Complete
        ↓
Read New Partitions
```

---

### (5) PhotonColumnarToRow

```text
PhotonColumnarToRow
```

👉 Photon engine data ko internally columnar format mein store/process karta hai.

Result return karne se pehle Spark usse row format mein convert kar raha hai.

```text
Columnar Format
        ↓
Row Format
```

Interview mein usually is step par focus nahi karte.

---

### (6) PhotonResultStage

```text
PhotonResultStage
```

👉 Final output stage.

Ab repartitioned DataFrame ready hai.

```text
Final Result Prepared
```

---

### (7) AdaptiveSparkPlan

```text
Arguments: isFinalPlan=false
```

👉 Spark AQE (Adaptive Query Execution) use kar raha hai.

Abhi query execute nahi hui hai, sirf execution plan dikhaya gaya hai.

Isliye Spark bol raha hai:

```text
isFinalPlan=false
```

Meaning:

> Abhi initial plan hai. Runtime statistics milne ke baad main plan ko aur optimize kar sakta hoon.

Agar action run karoge:

```python
rep_rr.count()
```

ya

```python
rep_rr.show()
```

to AQE runtime par final plan bana sakta hai.

---

## Photon Explanation

```text
The query is fully supported by Photon.
```

👉 Good news 🚀

Matlab poori

In [0]:
out_path = "/Volumes/workspace/default/repartition_demo/repartition_6"

rep_rr.write.mode("overwrite").parquet(out_path)
# now Go to Catalog then volumne ... repartition_6 to see the partitions

## Why Did Spark Create Multiple Files After `repartition(6, "studio")`?

Suppose we run:

```python
df.repartition(6, "studio") \
  .write \
  .mode("overwrite") \
  .parquet("/Volumes/output")
```

After writing the data, Spark creates multiple files in the output folder.

---

## 1. Why Are There 6 `part-*` Files?

We explicitly created:

```python
repartition(6, "studio")
```

This means Spark redistributed the data into **6 partitions**.

During the write operation:

```python
.write.parquet(...)
```

Spark typically creates:

```text
1 Partition = 1 Output File
```

Therefore Spark generates:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
part-00005
```

Total files = **6**

because total partitions = **6**

---

## 2. Why Does the Numbering Start From 0?

Spark uses **zero-based indexing** for partitions.

```text
Partition 0
Partition 1
Partition 2
Partition 3
Partition 4
Partition 5
```

Therefore the files are named:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
part-00005
```

---

## 3. Understanding the File Name

Example:

```text
part-00000-tid-349306387449989083-2f93...
```

### `part-00000`

```text
part-00000
```

Represents:

> Data written by Partition 0

---

### `tid-349306387449989083`

```text
tid-349306387449989083
```

Represents:

> Task ID / Job Identifier generated by Spark

Used internally to uniquely identify write tasks.

---

### Random String at the End

```text
2f93...
```

Represents:

> A unique identifier (UUID)

Used for:

- Uniqueness
- Avoiding file name conflicts
- Fault tolerance

---

## 4. Why Are File Sizes Different?

Example:

```text
3.54 KB
3.22 KB
3.10 KB
2.89 KB
3.28 KB
3.27 KB
```

All files are not exactly the same size.

Reason:

```python
repartition(6, "studio")
```

uses **Hash Partitioning**.

```text
hash(studio) % 6
```

Different studio values may have different numbers of records.

Example:

```text
Disney = 100 rows
Marvel = 60 rows
Sony   = 40 rows
```

As a result:

- Some partitions receive more records
- Some partitions receive fewer records

Therefore output file sizes vary slightly.

---

## 5. What is the `_SUCCESS` File?

```text
_SUCCESS
```

This is a success marker created by Spark.

Meaning:

> The write operation completed successfully.

If this file exists:

✅ Write completed successfully

✅ No failures during commit

---

## 6. What is the `_started_*` File?

Example:

```text
_started_349306387449989083
```

This metadata file indicates:

> Spark started the write operation.

---

## 7. What is the `_committed_*` File?

Example:

```text
_committed_349306387449989083
```

This metadata file indicates:

> Spark successfully committed the output.

Flow:

```text
Start Writing
      ↓
Write Data
      ↓
Commit Output
```

---

## Visual Representation

```text
Movies Data
      ↓
repartition(6, "studio")
      ↓

Partition 0 ──► part-00000
Partition 1 ──► part-00001
Partition 2 ──► part-00002
Partition 3 ──► part-00003
Partition 4 ──► part-00004
Partition 5 ──► part-00005

      ↓

_SUCCESS
_started
_committed
```

---

## Interview Answer

> When Spark writes a DataFrame, it usually creates one output file per partition. Since the DataFrame was repartitioned into 6 partitions using `repartition(6, "studio")`, Spark generated 6 output files (`part-00000` to `part-00005`). The numbering starts from 0 because Spark uses zero-based partition indexing. Files such as `_SUCCESS`, `_started`, and `_committed` are metadata files used by Spark to track successful execution and commit status of the write operation.

## Understanding `repartition(5, "studio")`

### Scenario

Assume:

```text
Total Rows = 20
Unique Studio Values = 7

Disney
Marvel
Sony
Warner
Fox
Universal
Paramount
```

and we run:

```python
df.repartition(5, "studio")
```

---

## What Happens Internally?

Since a partitioning column (`studio`) is provided, Spark uses **Hash Partitioning**.

Spark calculates:

```python
hash(studio) % 5
```

for every row and assigns it to one of the 5 partitions.

### Example Distribution

*(Illustrative Example)*

```text
Disney     → hash % 5 = Partition 2
Marvel     → hash % 5 = Partition 1
Sony       → hash % 5 = Partition 3
Warner     → hash % 5 = Partition 0
Fox        → hash % 5 = Partition 4
Universal  → hash % 5 = Partition 2
Paramount  → hash % 5 = Partition 1
```

Result:

```text
Partition 0
-----------
Warner rows

Partition 1
-----------
Marvel rows
Paramount rows

Partition 2
-----------
Disney rows
Universal rows

Partition 3
-----------
Sony rows

Partition 4
-----------
Fox rows
```

---

## Important Concept

We have:

```text
7 Unique Studio Values
5 Partitions
```

Since the number of unique values is greater than the number of partitions:

```text
Multiple studio values can be assigned to the same partition.
```

For example:

```text
Disney     → Partition 2
Universal  → Partition 2
```

Both studio values may share the same partition.

---

## Same Studio Values Always Stay Together

Suppose the data contains:

```text
Disney
Disney
Disney
Disney
```

All Disney rows will go to the same partition because:

```python
hash("Disney") % 5
```

always returns the same partition number.

Example:

```text
Partition 2
-----------
Disney
Disney
Disney
Disney
```

This behavior is very useful for:

- `groupBy()`
- `join()`
- Aggregations

---

## How Many Output Files Will Be Created?

If we write the DataFrame:

```python
df.repartition(5, "studio") \
  .write \
  .parquet("/path")
```

Spark will typically create:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
```

Total files = **5**

because total partitions = **5**

---

## What If We Create More Partitions than Unique Values?

Example:

```python
df.repartition(10, "studio")
```

We now have:

```text
7 Unique Studio Values
10 Partitions
```

Some partitions may end up empty.

Example:

```text
Partition 0 → Data
Partition 1 → Empty
Partition 2 → Data
Partition 3 → Empty
Partition 4 → Data
Partition 5 → Data
Partition 6 → Empty
Partition 7 → Data
Partition 8 → Empty
Partition 9 → Data
```

This happens because no studio value hashes to those partitions.

---

## Visual Representation

```text
7 Studios
   ↓
Hash Partitioning
(hash(studio) % 5)
   ↓

Partition 0 → Warner
Partition 1 → Marvel, Paramount
Partition 2 → Disney, Universal
Partition 3 → Sony
Partition 4 → Fox
```

---

## Interview Answer

> When `repartition(n, column)` is used, Spark performs hash partitioning on the specified column. It calculates `hash(column) % n` and distributes the rows into `n` partitions. Rows with the same key value always go to the same partition. If there are more unique key values than partitions, multiple key values may be assigned to the same partition. For example, with 7 unique studio values and `repartition(5, "studio")`, Spark creates 5 partitions and distributes the 7 studios among them using the hash function.

## Benefits of Partitioning in PySpark

Partitioning is one of the most important concepts in Spark because Spark processes data **partition by partition** in parallel across multiple executors.

---

## 1. Increases Parallelism

Spark assigns partitions to executors and cores.

Example:

```text
100 Million Records
        ↓
10 Partitions
        ↓
10 Tasks Can Run in Parallel
```

More partitions generally allow Spark to utilize more CPU cores.

✅ Faster processing

✅ Better resource utilization

---

## 2. Improves Job Performance

Instead of processing the entire dataset on a single machine:

```text
100 GB Data
```

Spark splits it into partitions:

```text
Partition 1
Partition 2
Partition 3
...
Partition N
```

and processes them simultaneously.

✅ Reduced execution time

✅ Distributed processing

---

## 3. Reduces Shuffle Cost

If data is partitioned on a frequently used column:

```python
df.repartition(10, "department")
```

then operations such as:

```python
groupBy("department")
join(...)
```

become more efficient because related records are already colocated.

✅ Less network traffic

✅ Reduced shuffle overhead

---

## 4. Optimizes Join Performance

Suppose two DataFrames are partitioned by the same key:

```python
employees.repartition(10, "dept_id")

departments.repartition(10, "dept_id")
```

When joining:

```python
employees.join(departments, "dept_id")
```

Spark may move less data across executors.

✅ Faster joins

✅ Better scalability

---

## 5. Prevents Data Skew

Without proper partitioning:

```text
Partition 1 = 1000 rows
Partition 2 = 10 rows
Partition 3 = 20 rows
```

One executor becomes overloaded while others remain idle.

Proper repartitioning helps distribute data more evenly.

✅ Better load balancing

✅ Reduced straggler tasks

---

## 6. Controls the Number of Output Files

Example:

```python
df.repartition(5).write.parquet("/output")
```

Generally creates:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
```

Without controlling partitions, Spark may create too many small files.

✅ Better file management

✅ Improved read performance later

---

## 7. Better Memory Utilization

Instead of loading huge datasets into a single executor:

```text
100 GB
```

Spark processes smaller chunks (partitions).

✅ Lower memory pressure

✅ Reduced OutOfMemory errors

---

## 8. Enables Fault Tolerance

If one partition fails:

```text
Partition 7 Failed
```

Spark only recomputes that partition.

It does **not** need to recompute the entire dataset.

✅ Faster recovery

✅ Reliable distributed processing

---

## Real-Life Example

Suppose:

```text
1 Billion Records
```

Without partitioning:

```text
Single Task
     ↓
Very Slow
```

With partitioning:

```text
Partition 1
Partition 2
Partition 3
...
Partition 100

     ↓

100 Tasks Run in Parallel
```

Result:

✅ Much faster execution

✅ Better cluster utilization

---

## Interview Answer

> Partitioning in Spark divides data into smaller chunks that can be processed in parallel across the cluster. The main benefits are increased parallelism, faster execution, better resource utilization, reduced shuffle cost, improved join and aggregation performance, prevention of data skew, better memory management, control over output files, and improved fault tolerance.

- ## Problems Solved by `repartition()` in Spark

`repartition()` is used to redistribute data across partitions by performing a shuffle. It helps solve several common performance and scalability issues in Spark.

---

## 1. Data Skew (Uneven Data Distribution)

### Problem

Some partitions contain significantly more data than others.

```text
Partition 1 = 1,000,000 rows
Partition 2 = 1,000 rows
Partition 3 = 500 rows
```

As a result:

- One executor works much longer
- Other executors remain idle

### Solution

```python
df.repartition(10)
```

Spark redistributes data more evenly across partitions.

✅ Better load balancing

✅ Faster job completion

---

## 2. Low Parallelism

### Problem

Your cluster may have many CPU cores available, but the dataset has too few partitions.

Example:

```text
Cluster: 16 CPU Cores
DataFrame: 2 Partitions
```

Only 2 tasks can run in parallel.

### Solution

```python
df.repartition(16)
```

Spark can now execute more tasks concurrently.

✅ Better cluster utilization

✅ Improved performance

---

## 3. Unbalanced Partitions

### Problem

Some partitions are very large while others are very small.

```text
Partition 1 = 5 GB
Partition 2 = 100 MB
Partition 3 = 50 MB
```

This creates bottlenecks during execution.

### Solution

```python
df.repartition(20)
```

The data is redistributed across more balanced partitions.

✅ Improved task distribution

✅ Reduced execution bottlenecks

---

## 4. Too Many Small Files

### Problem

Data may be spread across hundreds or thousands of tiny files.

```text
500 Small Files
```

Small files increase metadata overhead and slow down reads.

### Solution

```python
df.repartition(20)
```

Data can be consolidated into fewer partitions before writing.

✅ Reduced small-file problem

✅ Better read performance

---

## 5. Slow Join Operations

### Problem

Join operations often require expensive shuffles.

```python
df1.join(df2, "studio")
```

### Solution

```python
df1 = df1.repartition(10, "studio")
df2 = df2.repartition(10, "studio")
```

Partitioning both DataFrames on the join key can improve data locality.

✅ More efficient joins

✅ Reduced shuffle overhead

---

## 6. Slow Aggregations

### Problem

Operations such as:

```python
df.groupBy("studio").count()
```

require records with the same key to be brought together.

### Solution

```python
df.repartition(10, "studio")
```

Rows with the same studio value are placed in the same partition.

✅ Better aggregation performance

✅ Less data movement

---

## 7. Controlling the Number of Output Files

### Problem

Writing data may produce too many or too few files.

### Solution

```python
df.repartition(5) \
  .write.parquet("/output")
```

Spark typically creates:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
```

✅ Better file management

✅ Easier downstream processing

---

## Visual Summary

```text
Before Repartition

Partition 1 = 1,000,000 rows
Partition 2 = 1,000 rows
Partition 3 = 500 rows

        ↓

repartition()

        ↓

Partition 1 = 200,000 rows
Partition 2 = 200,000 rows
Partition 3 = 200,000 rows
Partition 4 = 200,000 rows
Partition 5 = 200,000 rows
```

Result:

✅ Balanced workload

✅ Higher parallelism

✅ Better performance

---

## Interview Answer

> `repartition()` is used to redistribute data across partitions by performing a shuffle. It helps solve problems such as data skew, low parallelism, unbalanced partitions, excessive small files, slow joins, slow aggregations, and poor cluster resource utilization. By evenly distributing data across the cluster, repartition improves scalability and overall Spark job performance.

In [0]:
# Bad : Creates 1000 tiny files(in this case you have 1000 partitions)
# df.write.parquet("output/")
df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/repartition_demo/parquet_output"
)

In [0]:
# Good : Creates 10 resonably-sized files
df.coalesce(10).write.parquet("/Volumes/workspace/default/repartition_demo/coalesce_output")

## Interview Answer

> `coalesce()` is primarily used to reduce the number of partitions without performing a full shuffle. If the current number of partitions is already smaller than the requested number, `coalesce()` will not increase the partitions. To increase the number of partitions, `repartition()` should be used instead. For example, if a DataFrame has only 1 partition and we call `coalesce(10)`, Spark will still keep 1 partition, resulting in only 1 output file when the DataFrame is written to storage.

In [0]:
from pyspark.sql import functions as F

df.select(F.spark_partition_id()).distinct().show()

# to see the partitions


# df.rdd.getNumPartitions()
# This will not work in Databricks Serverless because RDD APIs is disabled.
# 0 it means 1 partition is there


In [0]:
print("Interview Questions and Answers")

# PySpark Interview Questions & Answers
## Explain Plan, Lazy Evaluation, Transformations, Repartition, Coalesce, Partitioning

---

# 1. What is explain() in PySpark?

### Answer

`explain()` is used to display the execution plan of a DataFrame query.

It helps developers understand:

- How Spark executes a query
- Join strategies being used
- Shuffle operations
- Predicate pushdown
- Partition pruning
- Query optimizations performed by Catalyst Optimizer

`explain()` is mainly used for debugging and performance tuning.

---

# 2. Does explain() execute the query?

### Answer

No.

`explain()` only displays the execution plan.

It does not trigger any Spark job and does not read the data.

```python
df.filter("age > 18").explain()
```

Only the execution plan is displayed.

No execution happens.

---

# 3. What are the different modes of explain()?

### Answer

#### Default Mode

```python
df.explain()
```

Shows Physical Plan.

---

#### Extended Mode

```python
df.explain("extended")
```

Shows:

- Parsed Logical Plan
- Analyzed Logical Plan
- Optimized Logical Plan
- Physical Plan

---

#### Formatted Mode

```python
df.explain("formatted")
```

Provides a structured and readable execution plan.

---

#### Cost Mode

```python
df.explain("cost")
```

Shows cost and optimizer statistics if available.

---

# 4. What is Parsed Logical Plan?

### Answer

Parsed Logical Plan is the unresolved version of the query.

Spark understands SQL syntax but has not yet verified tables and columns.

Example:

```python
df.select("name")
```

At this stage Spark has not validated whether the column exists.

---

# 5. What is Analyzed Logical Plan?

### Answer

Analyzed Logical Plan is the resolved logical plan.

Spark verifies:

- Tables exist
- Columns exist
- Data types are valid

This plan is ready for optimization.

---

# 6. What is Optimized Logical Plan?

### Answer

Catalyst Optimizer rewrites the query into a more efficient version.

Examples:

- Combine multiple filters
- Remove unnecessary columns
- Push filters to datasource
- Reorder operations

Goal:

Improve performance before execution.

---

# 7. What is Physical Plan?

### Answer

Physical Plan is the actual execution strategy selected by Spark.

Examples:

- Broadcast Hash Join
- Sort Merge Join
- Hash Aggregate
- File Scan

Spark selects the most efficient physical plan.

---

# 8. What is PhotonResultStage?

### Answer

PhotonResultStage is the final execution stage when Databricks Photon Engine is used.

Photon is a vectorized query engine that improves query performance.

Benefits:

- Faster execution
- Better CPU utilization
- Lower query latency

---

# 9. Why do we use explain()?

### Answer

Use explain() when:

✅ Job is running slowly

✅ Optimizing joins

✅ Checking shuffles

✅ Verifying predicate pushdown

✅ Verifying partition pruning

✅ Understanding Spark internals

✅ Learning execution plans

---

# 10. What is Lazy Evaluation?

### Answer

Spark delays execution of transformations until an action is called.

Instead of executing immediately, Spark builds a DAG.

Execution starts only after an action such as:

```python
show()
count()
collect()
write()
```

is called.

---

# 11. Why does Spark use Lazy Evaluation?

### Answer

Because it allows Spark to:

- Optimize queries
- Merge transformations
- Reduce shuffles
- Apply predicate pushdown
- Eliminate unnecessary work
- Create an optimized execution plan

Result:

Better performance.

---

# 12. What is a Transformation?

### Answer

A transformation creates a new DataFrame without executing the job.

Examples:

```python
filter()
select()
withColumn()
drop()
groupBy()
join()
```

Transformations are lazy.

---

# 13. What is an Action?

### Answer

An action triggers actual execution.

Examples:

```python
show()
count()
collect()
take()
write()
```

Actions execute all pending transformations.

---

# 14. What is DAG in Spark?

### Answer

DAG stands for Directed Acyclic Graph.

Spark stores all transformations as a DAG before execution.

Example:

```text
Read
 ↓
Filter
 ↓
Select
 ↓
GroupBy
 ↓
Show
```

Spark optimizes and executes this DAG.

---

# 15. What is Predicate Pushdown?

### Answer

Predicate Pushdown means Spark pushes filters to the data source.

Instead of:

```text
Read All Data
Apply Filter
```

Spark performs:

```text
Read Only Required Data
```

Example:

```python
df.filter("year = 2025")
```

This reduces I/O and improves performance.

---

# 16. What is Partition Pruning?

### Answer

Partition Pruning means Spark reads only relevant partitions.

Example:

```python
df.filter("year = 2025")
```

If data is partitioned by year, Spark reads only the 2025 partition.

Benefits:

- Less I/O
- Faster queries

---

# 17. What is a Narrow Transformation?

### Answer

A narrow transformation is one where each output partition depends on only one input partition.

No shuffle occurs.

---

# 18. Examples of Narrow Transformations

### Answer

```python
select()
filter()
withColumn()
drop()
map()
flatMap()
coalesce()
```

Usually no shuffle.

---

# 19. Why are Narrow Transformations faster?

### Answer

Because:

- No network communication
- No shuffle
- Same partition processing

Hence performance is better.

---

# 20. What is a Wide Transformation?

### Answer

A wide transformation is one where output partitions depend on multiple input partitions.

Data must be redistributed.

Shuffle occurs.

---

# 21. Examples of Wide Transformations

### Answer

```python
groupBy()
orderBy()
join()
distinct()
dropDuplicates()
repartition()
```

These generally require shuffle.

---

# 22. Why are Wide Transformations expensive?

### Answer

Because:

- Data moves across executors
- Network communication occurs
- Shuffle files are created

Hence execution is slower.

---

# 23. What is Shuffle?

### Answer

Shuffle is the process of redistributing data across partitions and executors.

Shuffle is one of the most expensive operations in Spark.

---

# 24. How can you identify a Shuffle in explain()?

### Answer

Look for:

```text
Exchange
```

Example:

```text
Exchange hashpartitioning(...)
```

This indicates shuffle.

---

# 25. What is repartition()?

### Answer

`repartition()` changes the number of partitions in a DataFrame.

It always performs a shuffle.

Therefore it is a wide transformation.

---

# 26. Can repartition() increase partitions?

### Answer

Yes.

Example:

```python
df.repartition(20)
```

Spark creates 20 partitions.

---

# 27. Can repartition() decrease partitions?

### Answer

Yes.

Example:

```python
df.repartition(5)
```

Spark redistributes data into 5 partitions.

---

# 28. Does repartition() always shuffle data?

### Answer

Yes.

Repartition always causes a full shuffle.

---

# 29. What partitioning is used by repartition(n)?

### Answer

Round Robin Partitioning.

Example:

```python
df.repartition(10)
```

Rows are distributed evenly.

---

# 30. What partitioning is used by repartition(n, column)?

### Answer

Hash Partitioning.

Example:

```python
df.repartition(10, "studio")
```

Spark uses:

```text
hash(studio) % 10
```

---

# 31. Why repartition before joins?

### Answer

Records with the same key are moved to the same partition.

Benefits:

- Less shuffle
- Better join performance

---

# 32. Why repartition before groupBy?

### Answer

Similar keys are colocated.

Benefits:

- Faster aggregation
- Reduced network traffic

---

# 33. What problems does repartition() solve?

### Answer

- Data skew
- Low parallelism
- Uneven partitions
- Slow joins
- Slow aggregations
- Small file issues

---

# 34. What is coalesce()?

### Answer

`coalesce()` reduces the number of partitions.

It attempts to avoid a full shuffle.

---

# 35. Can coalesce() increase partitions?

### Answer

No.

Example:

```python
df.coalesce(10)
```

If DataFrame has 2 partitions, Spark usually keeps 2 partitions.

To increase partitions use:

```python
repartition()
```

---

# 36. Why is coalesce faster than repartition?

### Answer

Because coalesce typically avoids shuffle.

Less data movement means better performance.

---

# 37. When should you use coalesce()?

### Answer

Use coalesce:

- Before writing data
- To reduce small files
- To reduce partitions

Example:

```python
df.coalesce(10)
```

---

# 38. Repartition vs Coalesce?

### Answer

### Repartition

- Increase or decrease partitions
- Always shuffle
- Wide transformation

### Coalesce

- Mainly decrease partitions
- Usually avoids shuffle
- Narrow transformation

---

# 39. Why does Spark create multiple output files?

### Answer

Spark writes one file per partition.

Example:

```python
df.repartition(6)
```

Output:

```text
part-00000
part-00001
part-00002
part-00003
part-00004
part-00005
```

Total files = 6

---

# 40. What is the _SUCCESS file?

### Answer

`_SUCCESS` is a marker file indicating that the write operation completed successfully.

---

# 41. What are _started and _committed files?

### Answer

These are metadata files used during the write process.

```text
_started_*
```

Indicates write started.

```text
_committed_*
```

Indicates write committed successfully.

---

# 42. How do you check current partitions?

### Answer

```python
df.select(
    F.spark_partition_id()
).distinct().show()
```

---

# 43. Why are partitions important?

### Answer

Partitions provide:

- Parallelism
- Better performance
- Better resource utilization
- Reduced memory pressure
- Faster recovery
- Distributed execution

---

# 44. How does partitioning improve performance?

### Answer

Partitioning allows Spark to process multiple chunks simultaneously across executors.

More partitions generally increase parallelism.

---

# 45. Explain Spark's fault tolerance mechanism.

### Answer

Spark maintains lineage information through the DAG.

If a partition is lost, Spark recomputes only the missing partition rather than the entire dataset.

---

# 46. What is Adaptive Query Execution (AQE)?

### Answer

AQE allows Spark to optimize execution plans during runtime based on actual statistics.

Benefits:

- Better join selection
- Better partition sizes
- Reduced shuffle overhead

---

# 47. What does isFinalPlan=false mean in explain()?

### Answer

It indicates AQE is enabled and Spark may further optimize the plan during execution.

The final plan is not yet generated.

---

# 48. What is the most common interview definition of Lazy Evaluation?

### Answer

Spark delays execution of transformations until an action is called. This allows Catalyst Optimizer to optimize the complete execution plan and improve performance through techniques such as predicate pushdown, filter merging, reduced shuffling, and efficient execution planning.

---

# Quick Revision

```text
explain()          -> Shows execution plan

Lazy Evaluation    -> Execute only after action

Transformation     -> No execution

Action             -> Triggers execution

Predicate Pushdown -> Filter at source

Partition Pruning  -> Read fewer partitions

Narrow             -> No Shuffle

Wide               -> Shuffle Required

Shuffle            -> Data Movement

repartition()      -> Always Shuffle

coalesce()         -> Reduce Partitions

Round Robin        -> repartition(n)

Hash Partitioning  -> repartition(n, col)

1 Partition        -> 1 Output File

_SUCCESS           -> Successful Write

AQE                -> Runtime Optimization

Photon             -> Fast Vectorized Engine
```